# Тестовое задание на Windows через Colab

Полная инструкция: GUIDE_RU.md в обновлённом ZIP. На Windows ничего из Python/CUDA/Ubuntu для этого сценария не устанавливается. Здесь используются NVIDIA GPU Google, Qwen2.5-1.5B-Instruct и два QLoRA-адаптера.

Сначала Runtime → Change runtime type → GPU. Выполняйте ячейки по порядку кнопкой ▶. Run all не используйте: данные, оценки и отчёт нужно проверить вручную. Этот стартовый notebook не содержит заранее выполненного обучения или выдуманных результатов.

## Загрузка проекта

Выберите один исходный ZIP комплекта. Для восстановления новой машины можно выбрать последний checkpoint ZIP. После обычного перезапуска Python эту ячейку повторно не запускайте: папка остаётся.

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile, shutil
project_dir = Path('/content/llm-style-task')
if project_dir.exists():
    raise RuntimeError('Папка проекта уже существует. Для продолжения выполните проверку/run ниже; не распаковывайте стартовый ZIP поверх результатов.')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert len(zip_names) == 1, 'Загрузите один ZIP проекта'
with tempfile.TemporaryDirectory(dir='/content') as temporary_dir:
    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall(temporary_dir)
    candidates = [p.parent for p in Path(temporary_dir).rglob('train.py') if (p.parent / 'common.py').is_file() and (p.parent / 'data/raw/test_inputs.jsonl').is_file()]
    assert len(candidates) == 1, 'ZIP должен содержать один проект с train.py, common.py и data/raw'
    shutil.copytree(candidates[0], project_dir)
print('Проект загружен:', project_dir)


## Проверка назначенной GPU

Должна появиться таблица NVIDIA GPU. Если GPU нет, сначала проверьте тип среды.

In [ ]:
%cd /content/llm-style-task
!python --version
!nvidia-smi


## Установка библиотек

Дождитесь окончания. Затем Runtime → Restart session / Перезапустить сеанс. Не выбирайте удаление среды. Продолжайте с проверки после перезапуска; ZIP заново не загружается.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-train.txt'], check=True)


## Проверка после перезапуска и определение run

Повторяйте эту ячейку после каждого перезапуска Python. Команда run запускает скрипт и останавливается при его ошибке. T4 автоматически использует FP16; BF16=False на T4 — нормально.

In [ ]:
%cd /content/llm-style-task
import subprocess, sys
from pathlib import Path
from google.colab import files
import torch, transformers, peft, bitsandbytes
def run(*arguments):
    subprocess.run([sys.executable, *arguments], check=True)
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('peft', peft.__version__)
print('bitsandbytes', bitsandbytes.__version__)
assert torch.cuda.is_available(), 'CUDA GPU не найдена: сначала включите облачную GPU'
print('GPU', torch.cuda.get_device_name(0))
print('VRAM GiB', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('BF16 supported', torch.cuda.is_bf16_supported())


## Подготовка данных

После команды скачайте DATA_REVIEW.md и прочитайте все 50 пар. Шесть предложенных правок targets документированы отдельно. Не добавляйте пять финальных test-текстов в обучение.

In [ ]:
run('prepare_data.py')


In [ ]:
files.download('/content/llm-style-task/DATA_REVIEW.md')


## Наивный baseline

Запускайте после проверки данных. Первый прогон скачает модель и закрепит её ревизию. Десять validation-ответов, без адаптеров.

In [ ]:
run('generate.py', '--mode', 'baseline', '--split', 'validation')


## Подробный промпт без адаптера

Контроль эффекта промпта на тех же validation-исходниках.

In [ ]:
run('generate.py', '--mode', 'prompt-only', '--split', 'validation')


## Обучение адаптера type1

Дождитесь сохранения adapters/type1. Не запускайте type2 параллельно.

In [ ]:
run('train.py', '--style', 'type1')


## Резервная копия после type1

Скачайте ZIP на Windows и дождитесь полного скачивания. Это промежуточный проект, а не финальная сдача.

In [ ]:
run('package_submission.py', '--output', '/content/checkpoint_type1.zip')
files.download('/content/checkpoint_type1.zip')


## Обучение адаптера type2

Второй адаптер обучается независимо, поверх исходной базы.

In [ ]:
run('train.py', '--style', 'type2')


## Резервная копия обоих адаптеров

In [ ]:
run('package_submission.py', '--output', '/content/checkpoint_both_adapters.zip')
files.download('/content/checkpoint_both_adapters.zip')


## Проверка adapted на validation

Будет создан REVIEW.md с тридцатью ответами трёх режимов. Следующий этап — ручная оценка на Windows.

In [ ]:
run('generate.py', '--mode', 'adapted', '--split', 'validation')
run('make_review.py')


## Скачивание REVIEW.md

Откройте файл в VS Code/Блокноте Windows. Заполните четыре оценки в каждой из 30 строк: факты 0/1, стиль/структура/язык 1–5. Не меняйте первые три столбца, сохраняйте имя REVIEW.md.

In [ ]:
files.download('/content/llm-style-task/REVIEW.md')


## Загрузка заполненной оценки

Выберите исправленный REVIEW.md с Windows. Затем посчитайте сводку следующей ячейкой.

In [ ]:
edited = files.upload()
assert 'REVIEW.md' in edited, 'Выберите файл с точным именем REVIEW.md'
Path('/content/llm-style-task/REVIEW.md').write_bytes(edited['REVIEW.md'])
print('REVIEW.md обновлён')


In [ ]:
run('review_scores.py')


## Финальные генерации

Выполняйте после выбора настройки по validation. Пять test-источников → два стиля → десять постов.

In [ ]:
run('generate.py', '--mode', 'adapted', '--split', 'test')


## Скачивание и проверка десяти постов

Сверьте все посты с таблицей фактов в GUIDE_RU.md. Запустите две ячейки скачивания по очереди.

In [ ]:
files.download('/content/llm-style-task/outputs_type1.txt')


In [ ]:
files.download('/content/llm-style-task/outputs_type2.txt')


## Создание отчёта и фиксация окружения

Отчёт использует настоящие ответы и вашу сводку оценок. Далее заполните человеческие выводы на Windows. После ручного заполнения make_results.py повторно не запускается.

In [ ]:
run('make_results.py')
with open('/content/llm-style-task/requirements-train-lock.txt', 'w', encoding='utf-8') as output:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=output, check=True)


In [ ]:
files.download('/content/llm-style-task/RESULTS.md')


## Загрузка заполненного отчёта

На Windows замените все [ЗАПОЛНИТЬ] реальными наблюдениями, временем и статусом серверного теста. Сохраните имя RESULTS.md и выберите его здесь. vLLM в этом notebook не устанавливается; серверный кейс описан в отчёте как план, если своего NVIDIA-сервера нет.

In [ ]:
edited = files.upload()
assert 'RESULTS.md' in edited, 'Выберите файл с точным именем RESULTS.md'
Path('/content/llm-style-task/RESULTS.md').write_bytes(edited['RESULTS.md'])
print('RESULTS.md обновлён')


## Итоговая проверка и упаковка

Создаются полный ZIP с адаптерами и небольшой ZIP для GitHub через браузер. Скрипт проверяет заполнение и структуру, а не смысловую верность постов.

In [ ]:
run('finalize_submission.py', '--output-dir', '/content')


## Скачать полный проект

Сохраните архив на Windows. Он содержит бинарные веса адаптеров.

In [ ]:
files.download('/content/submission.zip')


## Скачать файлы для GitHub

Распакуйте github_upload.zip на Windows и загрузите его содержимое в корень репозитория через сайт GitHub. Большие бинарные веса исключены и остаются в полном архиве.

In [ ]:
files.download('/content/github_upload.zip')


## Сохранить выполненный notebook

File → Download → Download .ipynb. Добавьте как executed_colab.ipynb в браузерный экспорт перед загрузкой на GitHub. Порядок публикации — в GUIDE_RU.md. Сдача: ссылка на репозиторий, outputs_type1.txt, outputs_type2.txt и RESULTS.md.